In [ ]:
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.documents import Document
from pydantic import BaseModel, Field
from langchain_core.messages import AIMessage

#-------------------------------------------------------------------
from langchain_core.prompts import ChatPromptTemplate
from langchain_huggingface import HuggingFaceEmbeddings
from sentence_transformers import CrossEncoder
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain.retrievers.document_compressors import CrossEncoderReranker
from langchain.retrievers import ContextualCompressionRetriever
from langchain_chroma import Chroma
from transformers import AutoModel
#-------------------------------------------------------------------

from typing import Dict, Any, List, Tuple, Literal, Optional, Callable
import json
import copy
from tqdm import tqdm
import re
import textwrap
import asyncio
import numpy as np

from rag_info_extractor.llm_connector import OllamaLLM
from rag_info_extractor import embedding_server

### Test for ingest document

In [55]:
# -------------------------------- Azienda Name ---------------------------------
class _AziendaNameJSON(BaseModel):
    azienda_name: str = Field(description="nome della società a cui si riferisce il documento.")

def _get_name_azienda(page_content: str, llm_model: str) -> _AziendaNameJSON:
    """Extract Azienda name from the doc using LLM"""

    llm = OllamaLLM(llm_model=llm_model, temperature=0)

    template = textwrap.dedent("""\
        Sei un estrattore di entità. Dal testo seguente, individua la DENOMINAZIONE SOCIALE (nome completo ufficiale) della società.
        Regole:
        - Includi la forma giuridica (S.r.l., S.p.A., ecc.) se presente.
        - Rimuovi virgolette esterne; non aggiungere spiegazioni.
        - Se più società sono citate, scegli il soggetto principale del documento; se ambiguo, la prima denominazione completa.
        - Se assente, restituisci esattamente 'NON HO TROVATO'. 
        - Restituisci SOLO la DENOMINAZIONE SOCIALE (nome completo ufficiale) della società trovata.

        TESTO:
        {text}
    """)

    prompt = template.replace("{text}", page_content)

    ai_msg: _AziendaNameJSON = llm.ainvoke(
        output_format = "structured",
        memory = prompt,
        cache = False,
        info_schema = _AziendaNameJSON
    ) # type: ignore   

    return ai_msg 

# -------------------------------- Sede ---------------------------------
class _AziendaSedeJSON(BaseModel):
    azienda_sede: str = Field(description="sede della società a cui si riferisce il documento.")

def _get_sede_azienda(page_content: str, llm_model: str) -> _AziendaSedeJSON:
    """Extract Azienda name from the doc using LLM"""

    llm = OllamaLLM(llm_model=llm_model, temperature=0)

    template = textwrap.dedent("""\
        Sei un estrattore di entità.  
        Dal testo seguente, individua la SEDE LEGALE della società.

        Regole:
        - Riporta l’indirizzo completo così come appare nel testo (via, numero civico, città, CAP, provincia se presenti).
        - Non aggiungere né modificare nulla; mantieni l’ordine e la forma originale.
        - Se nel testo sono presenti più sedi, scegli la SEDE LEGALE principale; se non è chiaro, scegli la prima menzionata.
        - Se la sede non è indicata, restituisci esattamente 'NON HO TROVATO'.
        - Restituisci SOLO la SEDE LEGALE (nessun altro testo o spiegazione).

        TESTO:
        {text}
    """)

    prompt = template.replace("{text}", page_content)

    ai_msg: _AziendaSedeJSON = llm.ainvoke(
        output_format = "structured",
        memory = prompt,
        cache = False,
        info_schema = _AziendaSedeJSON
    ) # type: ignore

    return ai_msg

# async function to extract azienda Metadata
async def _get_metadata_azienda(
    page_content: str,
    llm_model: str,
    extract_metadata_functions: List[Callable] = [
        _get_name_azienda,
        _get_sede_azienda
    ]
):
    tasks = [f(page_content, llm_model) for f in extract_metadata_functions]
    results = await asyncio.gather(*tasks)

    return results


In [18]:
content = textwrap.dedent("""\
    Art.1.- E' costituita la società a responsabilità limita denominata
    "2KIND SRL"
    Art.22.- L'esercizio sociale inizia il primo gennaio e si chiude il trentuno
    dicembre di ogni anno.

    Allegato "B" dell'atto a Racc.2963
    S T A T U T O
    DENOMINAZIONE - SEDE - DURATA - OGGETTO
                          
    Art.2.- La Società ha sede in Roma.
    Le variazioni di indirizzo nell'ambito dello stesso comune, pur non
    costituendo modifica del presente statuto, sono adottate con decisione
    dell'organo amministrativo ovvero dei soci e sono comunicate al registro
    delle imprese competente.
    La società ha facoltà di istituire, ovunque lo ritenga necessario, uffici
    agenzie succursali o trasferirle e sopprimerle.

    Art.3.- La durata della società è fissata fino al trentuno dicembre
    duemilasessanta.
    Con delibera dell'Assemblea dei soci, potrà essere sciolta anticipatamente
    o prorogata.
""")
output = await _get_metadata_azienda(
    page_content = content,
    llm_model = "llama3.2:3b-instruct-q4_0"
)

In [64]:
for m in output[:1]:
    azienda_name = m.model_dump().get("azienda_name")

azienda_name

'2KIND SRL'

In [51]:
metadatas = {}
for m in output:
    metadatas.update(m.model_dump())
print(metadatas)

{'azienda_name': '2KIND SRL', 'azienda_sede': 'Via A. Rossi, 1, 00198 Roma, Italia'}


In [13]:
# -------------------------------- Azienda Name ---------------------------------
class _AziendaNameJSON(BaseModel):
    azienda_name: str = Field(description="nome della società a cui si riferisce il documento.")

def _get_name_azienda(page_content: str, llm_model: str) -> str:
    """Extract Azienda name from the doc using LLM"""

    llm = OllamaLLM(llm_model=llm_model, temperature=0)

    template = textwrap.dedent("""\
        Sei un estrattore di entità. Dal testo seguente, individua la DENOMINAZIONE SOCIALE (nome completo ufficiale) della società.
        Regole:
        - Includi la forma giuridica (S.r.l., S.p.A., ecc.) se presente.
        - Rimuovi virgolette esterne; non aggiungere spiegazioni.
        - Se più società sono citate, scegli il soggetto principale del documento; se ambiguo, la prima denominazione completa.
        - Se assente, restituisci esattamente 'NON HO TROVATO'. 
        - Restituisci SOLO la DENOMINAZIONE SOCIALE (nome completo ufficiale) della società trovata.

        TESTO:
        {text}
    """)

    prompt = template.replace("{text}", page_content)

    ai_msg: _AziendaNameJSON = llm.invoke(
        output_format = "structured",
        memory = prompt,
        cache = False,
        info_schema = _AziendaNameJSON
    ) # type: ignore   
    
    name = ai_msg.azienda_name

    return name

# -------------------------------- Sede ---------------------------------
class _AziendaSedeJSON(BaseModel):
    azienda_sede: str = Field(description="sede della società a cui si riferisce il documento.")

def _get_sede_azienda(page_content: str, llm_model: str) -> str:
    """Extract Azienda name from the doc using LLM"""

    llm = OllamaLLM(llm_model=llm_model, temperature=0)

    template = textwrap.dedent("""\
        Sei un estrattore di entità.  
        Dal testo seguente, individua la SEDE LEGALE della società.

        Regole:
        - Riporta l’indirizzo completo così come appare nel testo (via, numero civico, città, CAP, provincia se presenti).
        - Non aggiungere né modificare nulla; mantieni l’ordine e la forma originale.
        - Se nel testo sono presenti più sedi, scegli la SEDE LEGALE principale; se non è chiaro, scegli la prima menzionata.
        - Se la sede non è indicata, restituisci esattamente 'NON HO TROVATO'.
        - Restituisci SOLO la SEDE LEGALE (nessun altro testo o spiegazione).

        TESTO:
        {text}
    """)

    prompt = template.replace("{text}", page_content)

    ai_msg: _AziendaSedeJSON = llm.invoke(
        output_format = "structured",
        memory = prompt,
        cache = False,
        info_schema = _AziendaSedeJSON
    ) # type: ignore

    sede = ai_msg.azienda_sede

    return sede

# async function to extract azienda Metadata
def _get_metadata_azienda(
    page_content: str,
    llm_model: str,
    extract_metadata_functions: List[Callable] = [
        _get_name_azienda,
        _get_sede_azienda
    ]
):
    tasks = [f(page_content, llm_model) for f in extract_metadata_functions]

    return tasks
    # return {
    #     "azienda_name": results[0]
    # }

### Test for reranker/pruner models

In [9]:
RERANKER_MODEL = "D:/Users/yye7607/.cache/huggingface/hub/models--BAAI--bge-reranker-v2-m3/snapshots/953dc6f6f85a1b2dbfca4c34a2796e7dde08d41e"

re_ranker = CrossEncoder(RERANKER_MODEL, device="cpu", max_length=512) 

query = "Qual è la durata della società (fino a quale data)?"
context_candidates = [
    "\n        Art.3.- La durata della società è fissata fino al trentuno dicembre\n        duemilasessanta.\n        Con delibera dell'Assemblea dei soci, potrà essere sciolta anticipatamente\n        o prorogata.\n        ",
    "\n        BILANCIO E UTILI\n        Art.23.- Gli utili netti, in base a delibera assembleare, sono ripartiti come\n        segue:\n        - il cinque per cento (5%) sarà destinato alla riserva legale fino al\n        raggiungimento dell'importo pari al venti per cento del capitale sociale;\n        - la rimanenza è ripartita fra i soci in proporzione delle rispettive quote di\n        capitale, salvo che essi non decidano diversamente.\n        ",
    "\n        L'Assemblea deve essere convocata almeno una volta l'anno per\n        l'approvazione del bilancio, entro centoventi giorni dalla chiusura\n        dell'esercizio sociale, oppure ove la società sia tenuta alla redazione del\n        bilancio consolidato ovvero quando lo richiedano particolari esigenze\n        relative alla struttura ed all’oggetto della società, entro centottanta giorni\n        dalla sopradetta chiusura; in questi casi gli amministratori segnalano nella\n        relazione prevista dall’art. 2428 del codice civile le ragioni della dilazione.\n        ",

]
pairs = [(query, c) for c in context_candidates]


In [10]:
scores = re_ranker.predict(pairs, batch_size=2, show_progress_bar=False)

In [11]:
scores_with_idxs = [(i, round(score, 3)) for i, score in enumerate(scores)]

In [12]:
scores_with_idxs

[(0, np.float32(0.991)), (1, np.float32(0.001)), (2, np.float32(0.028))]

In [2]:
EMBEDDING_MODEL_NAME = "D:/Users/yye7607/.hf_models/embedding_models/e5-large-instruct"
embedding_func = HuggingFaceEmbeddings(
    model_name = EMBEDDING_MODEL_NAME, # HuggingFace embedding model
    encode_kwargs = {"normalize_embeddings": True}
)
vector_store = Chroma(
    embedding_function = embedding_func,
    persist_directory = "D:/Users/yye7607/Documents/work/Stage Amjad Ali/RAG/rag_information_extractor/data/vector_dbs/temp/custom_chunks",
    collection_name = "pdf_chunks"
)

In [3]:
RERANKER_MODEL = "D:/Users/yye7607/.cache/huggingface/hub/models--BAAI--bge-reranker-v2-m3/snapshots/953dc6f6f85a1b2dbfca4c34a2796e7dde08d41e"
fast_reranker = CrossEncoderReranker(model=HuggingFaceCrossEncoder(model_name=RERANKER_MODEL), top_n=8)
retriever = vector_store.as_retriever()
compression_retriever = ContextualCompressionRetriever(
            base_compressor=fast_reranker, base_retriever=retriever
            )

In [4]:
query = "Agli amministratori spetta il rimborso"
compression_retriever.invoke(query)

[Document(id='0d290c5c-68c0-4330-8bf3-36640fe144bd', metadata={'parent_id': 19, 'header': 'Art.19.– Agli Amministratori spetta, oltre al rimborso delle spese', 'total_pages': 7, 'trapped': '', 'format': 'PDF 1.5', 'author': '', 'start': 14830, 'pattern_name': 'art_keyword', 'title': 'Agli Amministratori spetta, oltre al rimborso delle spese', 'source': '8048909650002.pdf', 'producer': 'OAPDFPrinter        ', 'modDate': "D:20230711100553+02'00'", 'chunk_id': 17, 'end': 14981, 'keywords': '', 'child_id': 17, 'filename': '8048909650002.pdf', 'azienda': '2kind srl', 'creationDate': "D:20230711100553+02'00'", 'subject': '', 'creator': ''}, page_content='Art.19.– Agli Amministratori spetta, oltre al rimborso delle spese\nsostenute in ragione del loro ufficio, un compenso eventuale determinato dai\nsoci.'),
 Document(id='42284749-010a-43f7-9b5e-3889ac74ab07', metadata={'char_end': 559, 'pattern_name': 'fixed_size', 'format': 'PDF 1.5', 'modDate': "D:20230705154243+02'00'", 'creationDate': "D:

In [2]:
PRUNER_MODEL = "D:/Users/yye7607/.cache/huggingface/hub/models--naver--xprovence-reranker-bgem3-v1/snapshots/dd707795cc5a6ef5570f9be5282dab615fb7373d" 
pruner = AutoModel.from_pretrained(
    PRUNER_MODEL,
    trust_remote_code=True,
    local_files_only=True
)


In [3]:
query = "Qual è la durata della società (fino a quale data)?"
ori_context = [
    "\n        Art.3.- La durata della società è fissata fino al trentuno dicembre\n        duemilasessanta.\n        Con delibera dell'Assemblea dei soci, potrà essere sciolta anticipatamente\n        o prorogata.\n        ",
    "\n        BILANCIO E UTILI\n        Art.23.- Gli utili netti, in base a delibera assembleare, sono ripartiti come\n        segue:\n        - il cinque per cento (5%) sarà destinato alla riserva legale fino al\n        raggiungimento dell'importo pari al venti per cento del capitale sociale;\n        - la rimanenza è ripartita fra i soci in proporzione delle rispettive quote di\n        capitale, salvo che essi non decidano diversamente.\n        ",
    "\n        L'Assemblea deve essere convocata almeno una volta l'anno per\n        l'approvazione del bilancio, entro centoventi giorni dalla chiusura\n        dell'esercizio sociale, oppure ove la società sia tenuta alla redazione del\n        bilancio consolidato ovvero quando lo richiedano particolari esigenze\n        relative alla struttura ed all’oggetto della società, entro centottanta giorni\n        dalla sopradetta chiusura; in questi casi gli amministratori segnalano nella\n        relazione prevista dall’art. 2428 del codice civile le ragioni della dilazione.\n        ",

]

In [9]:
pruner_output = pruner.process([query], [ori_context], reorder=True, top_k=5, threshold=0.05)

Pruning contexts...:   0%|          | 0/1 [00:00<?, ?it/s]

Pruning contexts...: 100%|██████████| 1/1 [00:02<00:00,  2.42s/it]


In [10]:
pruner_output.get("pruned_context")[0]

["Art.3.- La durata della società è fissata fino al trentuno dicembre duemilasessanta. Con delibera dell'Assemblea dei soci, potrà essere sciolta anticipatamente o prorogata.",
 "L'Assemblea deve essere convocata almeno una volta l'anno per l'approvazione del bilancio, entro centoventi giorni dalla chiusura dell'esercizio sociale, oppure ove la società sia tenuta alla redazione del bilancio consolidato ovvero quando lo richiedano particolari esigenze relative alla struttura ed all’oggetto della società, entro centottanta giorni dalla sopradetta chiusura; in questi casi gli amministratori segnalano nella relazione prevista dall’art. 2428 del codice civile le ragioni della dilazione.",
 "BILANCIO E UTILI Art.23.- Gli utili netti, in base a delibera assembleare, sono ripartiti come segue: - il cinque per cento (5%) sarà destinato alla riserva legale fino al raggiungimento dell'importo pari al venti per cento del capitale sociale; - la rimanenza è ripartita fra i soci in proporzione delle 

### Test for embedding server

In [55]:
from rag_info_extractor import embedding_server

In [8]:
query = "Qual è la durata della società (fino a quale data)?"
context_candidates = [
    "\n        Art.3.- La durata della società è fissata fino al trentuno dicembre\n        duemilasessanta.\n        Con delibera dell'Assemblea dei soci, potrà essere sciolta anticipatamente\n        o prorogata.\n        ",
    "\n        BILANCIO E UTILI\n        Art.23.- Gli utili netti, in base a delibera assembleare, sono ripartiti come\n        segue:\n        - il cinque per cento (5%) sarà destinato alla riserva legale fino al\n        raggiungimento dell'importo pari al venti per cento del capitale sociale;\n        - la rimanenza è ripartita fra i soci in proporzione delle rispettive quote di\n        capitale, salvo che essi non decidano diversamente.\n        ",
    "\n        L'Assemblea deve essere convocata almeno una volta l'anno per\n        l'approvazione del bilancio, entro centoventi giorni dalla chiusura\n        dell'esercizio sociale, oppure ove la società sia tenuta alla redazione del\n        bilancio consolidato ovvero quando lo richiedano particolari esigenze\n        relative alla struttura ed all’oggetto della società, entro centottanta giorni\n        dalla sopradetta chiusura; in questi casi gli amministratori segnalano nella\n        relazione prevista dall’art. 2428 del codice civile le ragioni della dilazione.\n        ",

]

In [56]:
embedding_func_local = embedding_server.EmbedRequest(
    model_name = EMBEDDING_MODEL_NAME,
    encode_kwargs = {"normalize_embeddings": True}
)
embedded_query = embedding_func_local.embed_query(query)
print(embedded_query)

AttributeError: 'EmbedRequest' object has no attribute 'embed_query'

In [62]:
embedding_func_local.model_name

AttributeError: 'EmbedRequest' object has no attribute 'model_name'

In [52]:
EMBEDDING_MODEL_NAME = "D:/Users/yye7607/.hf_models/embedding_models/e5-large-instruct"
embedding_func = HuggingFaceEmbeddings(
    model_name = EMBEDDING_MODEL_NAME, # HuggingFace embedding model
    encode_kwargs = {"normalize_embeddings": True}
)
vector_store = Chroma(
    embedding_function = embedding_func,
    persist_directory = "D:/Users/yye7607/Documents/work/Stage Amjad Ali/RAG/rag_information_extractor/data/vector_dbs/temp/custom_chunks",
    collection_name = "pdf_chunks"
)
retriever = vector_store.as_retriever()

In [ ]:
retriever.invoke()

In [35]:
embedded_query_2 = embedding_func.embed_query(context_candidates[2])

In [36]:
v1 = np.array(embedded_query.get("embeddings", [])[0])
v2 = np.array(embedded_query_2)
np.linalg.norm(v2 - v1, 2)

np.float64(0.0)

In [ ]:
class ClassA(BaseModel):
    prop_1: str = Field(default="Ciao", alias="property_1")
    prop_2: str = Field(default="Mondo", alias="property_2")

    def __init__(self, **kwargs):
        super.__init__(**kwargs)
    